In [99]:
# Install packages
# %pip install requests
# %pip install bs4
# %pip install googlesearch-python
# %pip install stopit


Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for stopit: filename=stopit-1.1.2-py3-none-any.whl size=11952 sha256=a30ddbb73b7a86959426a7fc9ab67d468facccb7fb9f7b9dda8655b555b656f0
  Stored in directory: /Users/modekurt/Library/Caches/pip/wheels/48/8c/93/3afb1916772591fe6bcc25cdf8b1c5bdc362f0ec8e2f0fd413
Successfully built stopit
Note: you may need to restart the kernel to use updated packages.


In [100]:
import requests
from bs4 import BeautifulSoup
from googlesearch import search
import os
import pandas as pd
import numpy
from stopit import threading_timeoutable as timeoutable

In [101]:
planpath = 'plans/'

In [103]:
#Now search on google for climate action plans using extracted link

@timeoutable()
def download_search_results(query, city_name, city_code, country_iso, num_results):
    # Perform Google search and get the first 'num_results' URLs
    search_results = list(search(query, num_results=num_results, sleep_interval=2))
    print(search_results)

    # Create a directory to save the PDFs
    folder_path = planpath+country_iso+'/'+city_code+'-'+city_name
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    #collect info on downloads
    links, names = [], []

    # Download content from each URL
    for i, url in enumerate(search_results):
        try:
            response = requests.get(url)

            # Save the content to a file (you can modify this part based on your needs)
            if url.endswith('.pdf'):
                pdf_name = os.path.join(folder_path, city_code+'-[date]-'+os.path.basename(url))

                with open(pdf_name, 'wb') as pdf_file:
                    pdf_file.write(response.content)
                    print(f"{i + 1},{city_code},{pdf_name},{url}")

                    links.append(url)
                    names.append(pdf_name)
        except Exception as e:
            print(f"Error downloading result {i + 1}: {url}\nError: {e}")

    return links, names

# Example usage
# search_query = 'Mendoza' + 'climate adaptation plan' + 'filetype:pdf'
# download_search_results(search_query, 'sacramento', '12345', num_results=1)

In [ ]:
#Automate search of plans for all cities 

#load city list and list of searches in multiple languages 
city_list = pd.read_csv('output/missing_cities-mod.csv')
lang_list = pd.read_csv('data/Multilingual Search List.csv')
type_a_search_en = 'climate|"climate change" adaptation|protection|resilience plan|strategy|roadmap|program|measures|framework'
results_num = 1 #number of search results we want to return

city_codes, plan_links, plan_names = [], [], []

def plan_search(row):
    city_lang = str(row['Language'])
    city_lang_code = str(row['Language Code'])
    city_name = row['City']
    country_iso = row['ISO_Code']
    
    lang_search = lang_list.loc[lang_list['Code'] == city_lang_code, 'Plan Type A Search (adaptation plans)']
    #if the language doesn't exist in our search list, set the search to english
    type_a_search =  type_a_search_en if lang_search.isnull().any() else lang_search.values[0]
    query = city_name + ' ' + type_a_search + ' filetype:pdf'
    print(query)
    city_code = str(row['City Code'])
        
    #Download searches for city
    #Add 5 minute timeout when search get stuck due to captcha/429 Error
    links, names = download_search_results(query, city_name, city_code, country_iso, results_num, timeout=300)
    # links, names = ["test", "test"], ["b", "bla"]
    
    
    city_codes.extend([city_code]*len(links))
    plan_links.extend(links)
    plan_names.extend(names)
    #Maybe Document Link (in Google Drive)?
    return 

#look for searches for each city
try:
    searched_plans = city_list.apply(plan_search, axis=1)
except Exception as e:
    print(f'Error: {e} ')

downloaded_plans = pd.DataFrame({'City Code': city_codes, 'Plan Link': plan_links, 'Plan Name': plan_names})

#Combine downloaded information with other relevant info like country, city name, etc
downloaded_plans['City Code'] = downloaded_plans['City Code'].astype(int)
downloaded_plans['Dataset Source'] = 'Automated Google Search'
downloaded_plans['Document Status'] = 'Check Download'

downloaded_plans = downloaded_plans.merge(city_list, on='City Code')
downloaded_plans = downloaded_plans[['Country', 'City', 'City Code', 'ISO_Code', 'Plan Name', 'Plan Name (English translation)', 'Plan Year', 'Plan Link', 'Document Status', 
                                     'Document Relevance', 'Document Language', 'Document Link', 'Attachment/ link and name of plan', 'Dataset Source', 'Organization Name', 'Municipal Link']]
#write output to file
downloaded_plans.to_csv("output/downloaded_plans.csv")